In [ ]:
%pip -q install pandas rouge-score bert-score tqdm

In [ ]:
from pathlib import Path
import pandas as pd, json, re, ast, math, numpy as np
from tqdm.auto import tqdm
from rouge_score import rouge_scorer

INPUT_DIR = Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUTPUT_DIR = INPUT_DIR / 'logsumm_rouge_bertscore_two_scopes'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_GLOB = '*test_predictions*.csv'
BERT_MODEL = 'microsoft/deberta-xlarge-mnli'  # change if GPU memory is limited
BERT_LANG = 'en'
BERT_BATCH_SIZE = 8
BERT_DEVICE = None  # None = automatic
SCOPES = ['full_content', 'summary_only']
files = sorted(INPUT_DIR.glob(FILE_GLOB))
print('Files:', *[f.name for f in files], sep='\n  ')
assert files, 'No prediction CSVs found: update INPUT_DIR'

In [ ]:
def first_json_object(raw, kind):
    """Find first COMPLETE summary-level object, not embedded event dictionaries."""
    if isinstance(raw, dict):
        candidates = [(raw, 'dict')]
    elif not isinstance(raw, str) or not raw.strip():
        return None, 'empty'
    else:
        candidates = []
        decoder = json.JSONDecoder()
        for m in re.finditer(r'\{', raw):
            try:
                obj, _ = decoder.raw_decode(raw[m.start():])
                if isinstance(obj, dict):
                    candidates.append((obj, 'json'))
            except (ValueError, TypeError):
                pass
        if not candidates:
            for m in re.finditer(r'\{', raw):
                try:
                    obj = ast.literal_eval(raw[m.start():].split('\n```')[0].strip())
                    if isinstance(obj, dict): candidates.append((obj, 'python_literal'))
                except (ValueError, SyntaxError, TypeError): pass
    expected = {'extractive': {'explanation', 'critical_events', 'context_events'},
                'abstractive': {'explanation', 'summary'},
                'hybrid': {'explanation', 'critical_events', 'context_events', 'analysis', 'interpretation'}}[kind]
    for obj, method in candidates:
        if expected.intersection(obj):
            return obj, method
    return None, 'no_complete_summary_object'

def clean_value(v):
    if v is None: return ''
    if isinstance(v, str): return v.strip()
    if isinstance(v, list): return '\n'.join(filter(None, (clean_value(x) for x in v)))
    if isinstance(v, dict):
        return '; '.join(f'{k}: {clean_value(x)}' for k,x in v.items() if clean_value(x))
    return str(v)

def normalize(obj, kind):
    if not isinstance(obj, dict): return None
    explanation = clean_value(obj.get('explanation'))
    if kind == 'abstractive':
        main = clean_value(obj.get('summary'))
    elif kind == 'extractive':
        main = '\n'.join(filter(None, [clean_value(obj.get('critical_events')), clean_value(obj.get('context_events'))]))
    else:
        main = '\n'.join(filter(None, [clean_value(obj.get('critical_events')), clean_value(obj.get('context_events')), clean_value(obj.get('analysis')), clean_value(obj.get('interpretation'))]))
    full = '\n'.join(filter(None, [explanation, main]))
    return {'summary_only':main, 'full_content':full}

def recover(raw, kind):
    obj, method = first_json_object(raw, kind)
    if obj is not None:
        out = normalize(obj, kind)
        if out and any(out.values()): return out, method
        # A complete, well-formed JSON response with empty fields is preserved
        # so valid empty lists are not confused with a parsing failure.
        if out is not None: return out, 'valid_empty_object'
    # Do not score incomplete JSON as a complete output even if its explanation is visible.
    return None, method

def partial_explanation_present(raw):
    return isinstance(raw,str) and bool(re.search(r'"explanation"\s*:\s*"', raw))


In [ ]:
records=[]
for file in files:
    df=pd.read_csv(file, low_memory=False)
    kind = next((k for k in ['extractive','abstractive','hybrid'] if f'student_{k}_raw' in df.columns), None)
    if not kind or 'GT' not in df:
        print('SKIP',file.name);continue
    mode = 'zero_shot' if 'zero_shot' in file.name.lower() else 'stratified' if 'stratified' in file.name.lower() else 'fine_tuned' if 'fine_tuned' in file.name.lower() else 'other'
    if mode=='other': print('WARNING: unknown inference mode for',file.name)
    for _,r in df.iterrows():
        raw = r.get(f'student_{kind}_raw')
        pred,pm=recover(raw,kind)
        ref,rm=recover(r.get('GT'),kind)
        rec={'file':file.name,'dataset':file.name.split('_w')[0],
             'window_size':int(re.search(r'_w(\d+)',file.name).group(1)) if re.search(r'_w(\d+)',file.name) else None,
             'summary_type':kind,'inference_mode':mode,'segment_id':r.get('segment_id'),
             'student_parse_method':pm,'teacher_parse_method':rm,
             'student_status':r.get(f'student_{kind}_status'),
             'student_original_error':r.get(f'student_{kind}_error'),
             'student_eos_reached':r.get('student_eos_reached'),
             'partial_explanation_present':partial_explanation_present(raw) and pred is None}
        for scope in SCOPES:
            rec[f'pred_{scope}']=pred[scope] if pred else ''
            rec[f'ref_{scope}']=ref[scope] if ref else ''
            if ref is None: status='invalid_teacher'
            elif pred is None: status='invalid_student'
            elif pred[scope] and ref[scope]: status='scorable'
            elif not pred[scope] and not ref[scope]: status='both_empty'
            else: status='one_sided_empty'
            rec[f'{scope}_pair_status']=status
        records.append(rec)
rows=pd.DataFrame(records)
print('Total input predictions:',len(rows))
display(rows.groupby(['inference_mode','summary_type','student_parse_method'],dropna=False).size().rename('count').reset_index())


In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
for scope in SCOPES:
    status = rows[f'{scope}_pair_status']
    for metric in ['rouge1','rouge2','rougeL']:
        for part in ['precision','recall','f1']:
            col = f'{scope}_{metric}_{part}'
            rows[col] = 0.0  # ALL prediction rows remain in the denominator
            rows.loc[status == 'both_empty', col] = 1.0  # explicit agreement convention
    ids = rows.index[status == 'scorable']
    for i in tqdm(ids, desc=f'ROUGE {scope}'):
        scores = scorer.score(rows.at[i,f'ref_{scope}'], rows.at[i,f'pred_{scope}'])
        for metric in scores:
            for part,attr in [('precision','precision'),('recall','recall'),('f1','fmeasure')]:
                rows.at[i,f'{scope}_{metric}_{part}'] = getattr(scores[metric],attr)
print('ROUGE complete (failure-inclusive)')

In [ ]:
# BERTScore: one reusable scorer and finite tokenizer limit to prevent OverflowError.
from bert_score import BERTScorer
import torch

BERT_MAX_LENGTH = 512
scorer_kwargs = {
    'model_type': BERT_MODEL,
    'lang': BERT_LANG,
    'batch_size': BERT_BATCH_SIZE,
    'rescale_with_baseline': False,
}
if BERT_DEVICE is not None:
    scorer_kwargs['device'] = BERT_DEVICE
bert_scorer = BERTScorer(**scorer_kwargs)
bert_scorer._tokenizer.model_max_length = BERT_MAX_LENGTH
print('BERTScore tokenizer max length:', bert_scorer._tokenizer.model_max_length)

for scope in SCOPES:
    status = rows[f'{scope}_pair_status']
    for part in ['precision','recall','f1']:
        col = f'{scope}_bertscore_{part}'
        rows[col] = 0.0  # invalid, incomplete, and one-sided empty pairs
        rows.loc[status == 'both_empty', col] = 1.0
    ids = rows.index[status == 'scorable'].tolist()
    for start in tqdm(range(0, len(ids), BERT_BATCH_SIZE), desc=f'BERTScore {scope}'):
        batch = ids[start:start+BERT_BATCH_SIZE]
        predictions = rows.loc[batch, f'pred_{scope}'].astype(str).tolist()
        references = rows.loc[batch, f'ref_{scope}'].astype(str).tolist()
        with torch.inference_mode():
            p, r, f = bert_scorer.score(predictions, references,
                                        batch_size=BERT_BATCH_SIZE, verbose=False)
        for part, vals in [('precision',p),('recall',r),('f1',f)]:
            rows.loc[batch, f'{scope}_bertscore_{part}'] = vals.detach().cpu().numpy()
print('BERTScore complete (failure-inclusive)')

In [ ]:
# Generate SEPARATE outputs for full content and summary-only.
# The per-segment scores are initialized for ALL input rows, so group mean
# always has the complete input prediction-row denominator.
group = ['dataset','window_size','summary_type','inference_mode']
base_cols = group + ['file','segment_id','student_parse_method','teacher_parse_method',
                     'student_status','student_original_error','student_eos_reached',
                     'partial_explanation_present']
for scope in SCOPES:
    status_col = f'{scope}_pair_status'
    metric_cols = [f'{scope}_{m}_{part}' for m in ['rouge1','rouge2','rougeL','bertscore']
                   for part in ['precision','recall','f1']]
    # All missing values in a computed metric are errors, not reasons to drop cases.
    assert not rows[metric_cols].isna().any().any(), f'Missing scores in {scope}'
    scored = rows[base_cols + [status_col, f'pred_{scope}', f'ref_{scope}'] + metric_cols].copy()
    scored.to_csv(OUTPUT_DIR/f'per_segment_{scope}_scores.csv', index=False)
    agg = rows.groupby(group, dropna=False).agg(
        total_rows=('file','size'), **{col:(col,'mean') for col in metric_cols}
    ).reset_index()
    counts = (rows.groupby(group+[status_col],dropna=False).size()
              .unstack(fill_value=0).reset_index())
    for s in ['scorable','both_empty','one_sided_empty','invalid_student','invalid_teacher']:
        if s not in counts: counts[s] = 0
    counts = counts[group+['scorable','both_empty','one_sided_empty','invalid_student','invalid_teacher']]
    counts = counts.rename(columns={s:f'{scope}_{s}' for s in
                      ['scorable','both_empty','one_sided_empty','invalid_student','invalid_teacher']})
    agg = agg.merge(counts, on=group, validate='one_to_one')
    assert (agg[[f'{scope}_{s}' for s in
        ['scorable','both_empty','one_sided_empty','invalid_student','invalid_teacher']]].sum(axis=1)
        == agg['total_rows']).all(), f'Coverage mismatch: {scope}'
    agg.to_csv(OUTPUT_DIR/f'aggregate_{scope}_scores.csv',index=False)
    print(f'{scope}: {len(agg)} experiments, {len(scored)} prediction rows')
    print('  invalid teacher refs:',(rows[status_col]=='invalid_teacher').sum())
    display(agg[[*group,'total_rows',f'{scope}_rougeL_f1',f'{scope}_bertscore_f1']].head())

rows[base_cols + [f'{s}_pair_status' for s in SCOPES]].to_csv(
    OUTPUT_DIR/'extraction_diagnostics.csv',index=False)
print('Saved files to:', OUTPUT_DIR)
print(*(p.name for p in sorted(OUTPUT_DIR.glob('*.csv'))),sep='\n  ')

In [ ]:
# Comparability check: experiment groups may contain distinct segment subsets.
coverage_ids=(rows.groupby(group,dropna=False)['segment_id']
              .agg(lambda x: set(x.dropna().astype(str))))
for (dataset,window), part in coverage_ids.groupby(level=[0,1]):
    sets=part.tolist()
    if sets and len(set(map(frozenset,sets)))>1:
        print(f'UNMATCHED TEST SUBSETS: {dataset} w{window}; '
              f'{len(set.intersection(*sets))} shared segment IDs across {len(sets)} experiment files')
print('NOTE: An end-to-end score here covers all rows PRESENT in each file, not missing input chunks.')


In [ ]:
# Download outputs together in Colab; otherwise find the ZIP in OUTPUT_DIR.parent
import zipfile
zip_path=OUTPUT_DIR.parent/'logsumm_rouge_bertscore_two_scopes.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob('*.csv'):z.write(p,arcname=p.name)
print(zip_path)
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError: pass